In [1]:
from src.embedder import Embedder
import numpy as np
import re
from sentence_transformers import util

paragraph = """Bronkopneumonia pada anak usia dini sering kali disebabkan oleh infeksi bakteri, virus, atau jamur yang menyerang saluran napas bagian bawah. Gejala yang umum meliputi demam tinggi, batuk berdahak, sesak napas, dan napas cepat. Pada kasus berat, dapat terjadi penurunan kesadaran dan kegagalan pernapasan. Penanganan medis yang tepat sangat penting untuk mencegah komplikasi lebih lanjut. Selain itu, pemberian asuhan gizi yang sesuai juga berperan dalam proses pemulihan pasien."""

/mnt/nas-hpg9/eleazartadeo/miniconda3/envs/RAGFramework/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sentences = re.split(r'(?<=[.!?]) +', paragraph)
for i, sentence in enumerate(sentences):
    print(f"kalimat ke-{i+1}: {sentence}")

kalimat ke-1: Bronkopneumonia pada anak usia dini sering kali disebabkan oleh infeksi bakteri, virus, atau jamur yang menyerang saluran napas bagian bawah.
kalimat ke-2: Gejala yang umum meliputi demam tinggi, batuk berdahak, sesak napas, dan napas cepat.
kalimat ke-3: Pada kasus berat, dapat terjadi penurunan kesadaran dan kegagalan pernapasan.
kalimat ke-4: Penanganan medis yang tepat sangat penting untuk mencegah komplikasi lebih lanjut.
kalimat ke-5: Selain itu, pemberian asuhan gizi yang sesuai juga berperan dalam proses pemulihan pasien.


In [3]:
embedder = Embedder()
sentence_with_embedders = []
for sentence in sentences:
    sentence_with_embedders.append(embedder.embed_text_with_values(sentence))

for swe in sentence_with_embedders:
    print(f"\nSentence:\n{swe["text"]}\nSentence Length: {len(swe["text"])}\nDims:\n{swe["embedding"]}\n")


Sentence:
Bronkopneumonia pada anak usia dini sering kali disebabkan oleh infeksi bakteri, virus, atau jamur yang menyerang saluran napas bagian bawah.
Sentence Length: 141
Dims:
[-0.020747647, -0.075680755, -0.005831367, -0.037022382, 0.04281469, 0.012593704, -0.033400133, 0.052051987, -0.034430556, 0.026292862, -0.04470666, 0.006687533, 0.0038465364, -0.052159738, -0.008098625, -0.021166872, -0.02179969, 0.018281188, 0.009858417, 0.03720375, 0.06274454, 0.0124747725, -0.008446292, -0.080523096, 0.031757742, 0.04361146, -0.025356837, 0.032594647, -0.038882043, 0.007643103, -0.0029199144, 0.04814873, -0.022885105, 0.11672196, -0.050388597, 0.024775725, -0.0026445142, -0.026948947, 0.01217463, -0.003279597, -0.022393752, 0.044796146, 0.046645187, 0.11219982, 0.015195564, 0.021192193, 0.038755458, 0.02328439, 0.0053096963, 0.023981556, 0.019062767, -0.024533262, -0.010780067, 0.0025935282, -0.0026593995, -0.028996564, 0.053922895, -0.007332178, -0.019003674, 0.08551912, -0.035673607, 0.

In [4]:
print(len(sentence_with_embedders[0]["embedding"]))

768


# Rumus Cosine Similarity
![Cosine Similarity](images/1LfW66-WsYkFqWc4XYJbEJg.png)

In [5]:
for i in range(len(sentence_with_embedders) - 1):
    sim = util.cos_sim(sentence_with_embedders[i]["embedding"], sentence_with_embedders[i+1]["embedding"])
    print(f"Similarity of sentence {i} with sentence {i+1} is {float(sim)}")

Similarity of sentence 0 with sentence 1 is 0.410122275352478
Similarity of sentence 1 with sentence 2 is 0.5140658020973206
Similarity of sentence 2 with sentence 3 is 0.3189830183982849
Similarity of sentence 3 with sentence 4 is 0.447454571723938


In [ ]:
chunks = []
chunk_size = 500
embeddings = embedder.embed_documents(sentences)
embeddings = np.array(embeddings)

current_chunk = [sentences[0]]
current_emb = embeddings[0]
i_chunk = 1
for i in range(1, len(sentences)):
    sim = util.cos_sim(current_emb, embeddings[i])
    print(f"Similarity between current chunk {i_chunk} and sentence {i+1}: {sim}")
    if sim < 0.5 or sum(len(s) for s in current_chunk) > chunk_size:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]
        current_emb = embeddings[i]
        print(f"Current chunk {i_chunk} and current sentence {i+1} not similar because {sim} is less than 0.5 or {sum(len(s) for s in current_chunk)} is over {chunk_size}, making a new chunk (Chunk {i_chunk + 1})")
        i_chunk += 1
    else:
        current_chunk.append(sentences[i])
        current_emb = (current_emb + embeddings[i]) / 2
        print(f"Current chunk {i} and current sentence {i+1} similar (similarity value is {sim}), append the sentence into the chunk, recalculate the embedder's value")

if current_chunk:
    chunks.append(" ".join(current_chunk))

Similarity between current chunk 1 and sentence 2: tensor([[0.4101]], dtype=torch.float64)
Current chunk 1 and current sentence 2 not similar because tensor([[0.4101]], dtype=torch.float64) is less than 0.5 or 85 is over 500, making a new chunk
Similarity between current chunk 2 and sentence 3: tensor([[0.5141]], dtype=torch.float64)
Current chunk 2 and current sentence 3 similar (similarity value is tensor([[0.5141]], dtype=torch.float64)), append the sentence into the chunk, recalculate the embedder's value
Similarity between current chunk 2 and sentence 4: tensor([[0.2913]], dtype=torch.float64)
Current chunk 2 and current sentence 4 not similar because tensor([[0.2913]], dtype=torch.float64) is less than 0.5 or 82 is over 500, making a new chunk
Similarity between current chunk 3 and sentence 5: tensor([[0.4475]], dtype=torch.float64)
Current chunk 3 and current sentence 5 not similar because tensor([[0.4475]], dtype=torch.float64) is less than 0.5 or 90 is over 500, making a new c

In [7]:
print("Final Chunks made:")
for i, chunk in enumerate(chunks):
    print(f"Chunk-{i+1}\t: {chunk}")


Final Chunks made:
Chunk-1	: Bronkopneumonia pada anak usia dini sering kali disebabkan oleh infeksi bakteri, virus, atau jamur yang menyerang saluran napas bagian bawah.
Chunk-2	: Gejala yang umum meliputi demam tinggi, batuk berdahak, sesak napas, dan napas cepat. Pada kasus berat, dapat terjadi penurunan kesadaran dan kegagalan pernapasan.
Chunk-3	: Penanganan medis yang tepat sangat penting untuk mencegah komplikasi lebih lanjut.
Chunk-4	: Selain itu, pemberian asuhan gizi yang sesuai juga berperan dalam proses pemulihan pasien.
